In [25]:
# from google.colab import drive
# drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
# !unzip exam.zip

Archive:  exam.zip
replace exam/exam/exam0/001_exam0_1.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [27]:
# !pip install ultralytics

In [28]:
import cv2
import numpy as np
import os
import glob
import pandas as pd
import torch
from torchvision import transforms, models
from ultralytics import YOLO

# ================= CẤU HÌNH HỆ THỐNG =================
YOLO_DETECT_PATH = '/content/drive/MyDrive/OMR-Datasets/train_detect/Yolo26s/OMR_Localization/yolo26s_omr_detect/weights/best.pt'
EFFICIENTNET_WEIGHTS_PATH = '/content/drive/MyDrive/OMR-Datasets/train-cls-v2/scene2/EfficientNetB0-v2/Fold_5/efficientnetb0_fold5_gan.pth'
IMAGE_FOLDER_PATH = '/content/exam/exam/exam5'
TEMPLATE_IMG = '/content/exam/ModelAnswer'
OUTPUT_FILE = 'ket_qua_cham_thi_hoan_thien.csv'
TARGET_EXAM_ID = "exam5"

# ================= DỮ LIỆU NGƯỜI DÙNG TỰ NHẬP =================
# Giả lập dữ liệu cấu hình do người dùng nhập vào hệ thống
USER_INPUT_METADATA = {
    "exam0": {
        1: { # Trang 1
            "image_name": f"{TEMPLATE_IMG}/exam0/modelAnswer_exam0_1.png",
            "questions_structure": [3]*4, # 4 câu, mỗi câu 3 đáp án
            "answer_key": {"Q1":"A", "Q2":"B", "Q3":"C", "Q4":"B"}
        },
        2: { # Trang 2
            "image_name": f"{TEMPLATE_IMG}/exam0/modelAnswer_exam0_2.png",
            "questions_structure": [3]*3, # 3 câu, mỗi câu 3 đáp án
            "answer_key": {"Q5":"B", "Q6":"A", "Q7":"B"}
        },
        3: { # Trang 3
            "image_name": f"{TEMPLATE_IMG}/exam0/modelAnswer_exam0_3.png",
            "questions_structure": [3]*6, # 6 câu, mỗi câu 4 đáp án
            "answer_key": {"Q8":"B", "Q9":"C", "Q10":"B", "Q11":"B", "Q12":"C", "Q13":"C"}
        }
    },

    "exam5": {
        1: { # Trang 1
            "image_name": f"{TEMPLATE_IMG}/exam5/modelAnswer_exam5_1.png",
            "questions_structure": [4]*10 + [2]*10, # 20 câu, 10 câu 4 đáp án, 10 câu 2 đáp án
            "answer_key": {"Q1":"A", "Q2":"B", "Q3":"C", "Q4":"B", "Q5":"B", "Q6":"B", "Q7":"B", "Q8":"B", "Q9":"B", "Q10":"B",
                           "Q11":"B", "Q12":"B", "Q13":"B", "Q14":"B", "Q15":"B", "Q16":"B", "Q17":"B", "Q18":"B", "Q19":"B", "Q20":"B",}
        }
    }
}


In [29]:
# ============================================
# 1. HÀM CĂN CHỈNH ẢNH
# ============================================
def align_images(im1, im2, max_features=5000, keep_percent=0.2):
    """
    Căn chỉnh ảnh im1 (Student) theo im2 (Template).
    Sử dụng đặc trưng ORB và Homography.
    """
    # Chuyển sang ảnh xám
    img1Gray = cv2.cvtColor(im1, cv2.COLOR_BGR2GRAY)
    img2Gray = cv2.cvtColor(im2, cv2.COLOR_BGR2GRAY)

    # Phát hiện đặc trưng ORB
    orb = cv2.ORB_create(max_features)
    keypoints1, descriptors1 = orb.detectAndCompute(img1Gray, None)
    keypoints2, descriptors2 = orb.detectAndCompute(img2Gray, None)

    if descriptors1 is None or descriptors2 is None:
        print("⚠ Không tìm thấy đặc trưng để căn chỉnh.")
        return None

    # So khớp đặc trưng
    matcher = cv2.DescriptorMatcher_create(cv2.DESCRIPTOR_MATCHER_BRUTEFORCE_HAMMING)
    matches = matcher.match(descriptors1, descriptors2, None)

    # Lọc lấy các điểm khớp tốt nhất
    matches = sorted(matches, key=lambda x: x.distance)
    keep = int(len(matches) * keep_percent)
    matches = matches[:keep]

    # Lấy tọa độ các điểm tương đồng
    points1 = np.zeros((len(matches), 2), dtype=np.float32)
    points2 = np.zeros((len(matches), 2), dtype=np.float32)

    for i, match in enumerate(matches):
        points1[i, :] = keypoints1[match.queryIdx].pt
        points2[i, :] = keypoints2[match.trainIdx].pt

    # Tính ma trận biến đổi (Homography)
    try:
        h, mask = cv2.findHomography(points1, points2, cv2.RANSAC)
        if h is None: return None

        # Biến đổi ảnh bài làm (Warp)
        height, width, channels = im2.shape
        aligned_img = cv2.warpPerspective(im1, h, (width, height))
        return aligned_img
    except Exception as e:
        print(f"⚠ Lỗi Homography: {e}")
        return None

In [30]:
# ============================================
# 2. HÀM SẮP XẾP TỌA ĐỘ
# ============================================
def sort_contours_grid(rects):
    """
    Sắp xếp các ô trắc nghiệm theo hàng (Row-by-Row) thông minh hơn.
    Tự động thích nghi với kích thước ô để tránh lỗi lệch dòng.
    """
    if not rects: return []

    # 1. Tính chiều cao trung bình của các ô
    # Để dùng làm ngưỡng phân dòng (Threshold).
    # Nếu 2 ô cách nhau ít hơn 1/2 chiều cao ô -> Coi là cùng dòng.
    heights = [r[3] for r in rects]
    avg_height = np.mean(heights)
    threshold_y = avg_height * 0.6  # Ngưỡng thích ứng (thường là 50-60% chiều cao)

    # 2. Sắp xếp sơ bộ theo Y để đi từ trên xuống dưới
    rects = sorted(rects, key=lambda b: b[1])

    sorted_rects = []
    current_row = []
    last_y = rects[0][1]

    for rect in rects:
        y = rect[1]
        # Kiểm tra chênh lệch Y so với ô đầu tiên của dòng hiện tại
        if abs(y - last_y) <= threshold_y:
            current_row.append(rect)
        else:
            # --- KẾT THÚC DÒNG CŨ ---
            # Sort các ô trong dòng cũ theo X (Trái -> Phải)
            current_row.sort(key=lambda b: b[0])
            sorted_rects.extend(current_row)

            # --- BẮT ĐẦU DÒNG MỚI ---
            current_row = [rect]
            last_y = y # Cập nhật Y chuẩn mới cho dòng này

    # Xử lý dòng cuối cùng
    if current_row:
        current_row.sort(key=lambda b: b[0])
        sorted_rects.extend(current_row)

    return sorted_rects

In [31]:
# ============================================
# 3. QUÉT TEMPLATE ĐỂ LẤY MAP TỌA ĐỘ
# ============================================
def get_template_layout(template_path, detect_model, questions_structure, answer_key):
    """
    Tự động xử lý thừa ô do nhận diện nhầm khung bao (Container).
    """
    print(f"--- Phân tích ảnh mẫu: {os.path.basename(template_path)} ---")
    img = cv2.imread(template_path)
    if img is None: raise ValueError("Không tìm thấy ảnh mẫu!")

    total_bubbles_needed = sum(questions_structure)
    print(f"   > Mục tiêu: Cần tìm {total_bubbles_needed} ô.")

    # Các ngưỡng conf thử dần
    try_confs = [0.5, 0.4, 0.3, 0.25, 0.15]

    found_rects = []

    for conf in try_confs:
        results = detect_model(img, conf=conf, verbose=False)

        raw_rects = []
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            w, h = x2 - x1, y2 - y1
            raw_rects.append([int(x1), int(y1), int(w), int(h)])

        if len(raw_rects) == 0: continue

        # ====================================================
        # BƯỚC 1: LỌC THEO DIỆN TÍCH (MEDIAN FILTER)
        # ====================================================
        areas = [r[2] * r[3] for r in raw_rects]
        median_area = np.median(areas)

        # Chỉ giữ lại box có diện tích 40% -> 220% so với median
        # (Nới lỏng cận trên xíu để tránh xóa nhầm, vì ta sẽ lọc kỹ ở Bước 2)
        step1_rects = []
        for r in raw_rects:
            area = r[2] * r[3]
            if area > 0.4 * median_area and area < 2.5 * median_area:
                step1_rects.append(r)

        # ====================================================
        # BƯỚC 2: LOẠI BỎ KHUNG BAO (CONTAINER SUPPRESSION) - QUAN TRỌNG
        # ====================================================
        # Logic: Nếu Box A chứa Box B bên trong -> Xóa Box A

        def is_containing(boxA, boxB):
            # Kiểm tra xem boxA có chứa boxB không
            xa, ya, wa, ha = boxA
            xb, yb, wb, hb = boxB

            # Tính tọa độ góc dưới phải
            xa2, ya2 = xa + wa, ya + ha
            xb2, yb2 = xb + wb, yb + hb

            # Box A chứa Box B nếu tọa độ B nằm gọn trong A (có dung sai 5px)
            padding = 5
            return (xa < xb + padding) and (ya < yb + padding) and \
                   (xa2 > xb2 - padding) and (ya2 > yb2 - padding)

        indices_to_remove = set()
        N = len(step1_rects)
        for i in range(N):
            for j in range(N):
                if i == j: continue
                # Nếu box i chứa box j -> Đánh dấu box i để xóa
                if is_containing(step1_rects[i], step1_rects[j]):
                    indices_to_remove.add(i)
                    break # Đã biết i là container thì break luôn

        final_rects = []
        for i in range(N):
            if i not in indices_to_remove:
                final_rects.append(step1_rects[i])

        print(f"   > Conf={conf}: Raw={len(raw_rects)} -> FilterArea={len(step1_rects)} -> FilterContainer={len(final_rects)}")

        found_rects = final_rects

        # Kiểm tra điều kiện dừng
        if len(found_rects) == total_bubbles_needed:
            print("   ✅ Đã tìm thấy ĐỦ số lượng ô!")
            break

    # ====================================================
    # CUỐI CÙNG: CHECK LẠI SỐ LƯỢNG
    # ====================================================
    if len(found_rects) != total_bubbles_needed:
        # Fallback: Nếu vẫn thừa, ta sẽ sort theo diện tích và lấy N ô nhỏ nhất (thường ô đáp án nhỏ hơn khung)
        if len(found_rects) > total_bubbles_needed:
             print("   ⚠ Vẫn thừa ô. Fallback: Lấy N ô có diện tích nhỏ nhất gần với Median.")
             # Tính lại khoảng cách tới median area
             areas = [r[2]*r[3] for r in found_rects]
             median_final = np.median(areas)
             # Sort theo độ chênh lệch diện tích so với median
             found_rects.sort(key=lambda r: abs((r[2]*r[3]) - median_final))
             found_rects = found_rects[:total_bubbles_needed]
             print(f"   -> Đã cắt gọt còn: {len(found_rects)} ô.")
        else:
            msg = (f"❌ Lỗi Detect: Cần {total_bubbles_needed}, tìm thấy {len(found_rects)}.\\n"
                   f"   Hãy kiểm tra lại ảnh mẫu hoặc giảm ngưỡng conf thấp hơn nữa.")
            raise ValueError(msg)

    # 4. Sắp xếp & Mapping (Giữ nguyên)
    sorted_rects = sort_contours_grid(found_rects)

    layout_map = []
    current_idx = 0
    # Lấy danh sách tên câu hỏi thực tế từ answer_key (VD: ['Q5', 'Q6', 'Q7'])
    question_names = list(answer_key.keys())
    for q_idx, num_opts in enumerate(questions_structure):
        q_name = question_names[q_idx] # Dùng đúng tên thay vì q_idx + 1
        options = [chr(65+k) for k in range(num_opts)]
        for opt in options:
            rect = sorted_rects[current_idx]
            layout_map.append({
                'label': f"{q_name}_{opt}", # Label giờ sẽ chuẩn Q5_A, Q8_C...
                'rect': rect
            })
            current_idx += 1

    return layout_map, img

In [32]:
# =============================================================================
# 4. HÀM CHẤM ĐIỂM LINH HOẠT
# =============================================================================

def grade_sheet_dynamic(yolo_results, answer_key):
    total_score = 0.0
    max_score = 0.0
    errors = []

    # Duyệt qua từng câu hỏi trong Đáp án chuẩn
    for q_name, correct_ans in answer_key.items():
        max_score += 1.0

        # Tìm tất cả các ô của câu hỏi này (Q1_A, Q1_B,...) trong kết quả YOLO
        # Lấy các key bắt đầu bằng "Q1_"
        related_keys = [k for k in yolo_results.keys() if k.startswith(f"{q_name}_")]

        confirmed_opts = []
        for key in related_keys:
            if yolo_results[key] == 'confirmed':
                # key là "Q1_A" -> lấy "A"
                opt_char = key.split('_')[1]
                confirmed_opts.append(opt_char)
        # In so sanh dap an
        # print(f"answer_key: {correct_ans} - answer: {len(confirmed_opts)} - {confirmed_opts}")

        # Logic chấm điểm
        if len(confirmed_opts) == 1:
            if confirmed_opts[0] == correct_ans:
                total_score += 1.0
            # else: Sai -> 0 điểm
        elif len(confirmed_opts) > 1:
            errors.append(f"{q_name}: Chọn nhiều ({','.join(confirmed_opts)})")
        # else: Không chọn -> 0 điểm

    return total_score, max_score, errors

In [33]:
# ================= KHỞI TẠO MÔ HÌNH =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Model Detect (YOLOv8)
detect_model = YOLO(YOLO_DETECT_PATH)

# 2. Model Classify (EfficientNet-B0 gốc Float32)
class OMR_Classifier:
    def __init__(self, model_path, device):
        self.device = device
        self.model = models.efficientnet_b0(weights=None)
        self.model.classifier[1] = torch.nn.Linear(self.model.classifier[1].in_features, 3)
        self.model.load_state_dict(torch.load(model_path, map_location=device))
        self.model.to(self.device)
        self.model.eval()

        # Dùng resize 128x128 như lúc train
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((128, 128)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        self.classes = ['confirmed', 'crossedout', 'empty']

    # THÊM MỚI: Thuật toán đắp viền trắng tạo ảnh vuông (SquarePad)
    def square_pad(self, image, pad_color=(255, 255, 255)):
        h, w = image.shape[:2]
        max_dim = max(h, w)
        top = (max_dim - h) // 2
        bottom = max_dim - h - top
        left = (max_dim - w) // 2
        right = max_dim - w - left
        # Đắp viền màu trắng (tránh làm nhiễu nét mực)
        padded = cv2.copyMakeBorder(image, top, bottom, left, right, cv2.BORDER_CONSTANT, value=pad_color)
        return padded

    def predict_batch(self, roi_images):
        if not roi_images:
            return []

        tensors = []
        for img in roi_images:
            # Bước 1: Đắp viền trắng thành ảnh vuông
            sq_img = self.square_pad(img)
            # Bước 2: Chuyển màu và Transform
            rgb_img = cv2.cvtColor(sq_img, cv2.COLOR_BGR2RGB)
            tensors.append(self.transform(rgb_img))

        batch = torch.stack(tensors).to(self.device)
        with torch.no_grad():
            outputs = self.model(batch)
            _, preds = torch.max(outputs, 1)
        return [self.classes[p.item()] for p in preds]

classify_model = OMR_Classifier(EFFICIENTNET_WEIGHTS_PATH, device)

In [38]:
def main():
    print("--- HỆ THỐNG CHẤM THI OMR (EFFICIENTNET-B0) ---")

    exam_config = USER_INPUT_METADATA.get(TARGET_EXAM_ID)
    if not exam_config: return

    # BƯỚC 1: KHỞI TẠO TEMPLATE
    template_cache = {}
    for page_num, info in exam_config.items():
        tpl_path = info["image_name"]
        print(f" >> Đang quét Bản đồ Layout Trang {page_num}...")
        # Lấy tọa độ bằng YOLO-detect
        layout, img_cv2 = get_template_layout(tpl_path, detect_model, info["questions_structure"], info["answer_key"])
        template_cache[page_num] = (layout, img_cv2)

    # BƯỚC 2: QUÉT ẢNH & CHẤM ĐIỂM
    image_paths = glob.glob(os.path.join(IMAGE_FOLDER_PATH, '*.*'))
    if len(image_paths) == 0:
        print(f"❌ KHÔNG TÌM THẤY ẢNH NÀO TRONG THƯ MỤC: {IMAGE_FOLDER_PATH}")
        return
    student_results = {}

    for img_path in image_paths:
        filename = os.path.basename(img_path)

        # Parse Tên file: SBD_MaDe_Trang.jpg (VD: 10299_exam0_1.jpg)
        try:
            name_parts = os.path.splitext(filename)[0].split('_')
            student_id = name_parts[0]
            exam_id = name_parts[1]
            page_num = int(name_parts[2])
        except Exception:
            print(f"Bỏ qua {filename} do sai định dạng tên file.")
            continue

        if exam_id != TARGET_EXAM_ID or page_num not in template_cache:
            continue

        # Đọc và Căn chỉnh ORB
        student_img = cv2.imread(img_path)
        if student_img is None:
            print(f"⚠ Bỏ qua file lỗi/không phải ảnh: {filename}")
            continue
        current_layout, current_tpl_img = template_cache[page_num]
        aligned_img = align_images(student_img, current_tpl_img)
        if aligned_img is None: continue

        # Cắt ROI (Padding 2px)
        roi_imgs, roi_keys = [], []


        for item in current_layout:
            x, y, w, h = item['rect']
            roi = aligned_img[y:y+h, x:x+w]

            if roi.size > 0:
                roi_imgs.append(roi)
                roi_keys.append(item['label'])
            else:
                print(f"⚠ Lỗi cắt ô {item['label']} bị tràn viền ở SBD: {student_id}")

        # Phân loại bằng EfficientNet-B0
        preds = classify_model.predict_batch(roi_imgs)

        student_answers = {roi_keys[i]: preds[i] for i in range(len(preds))}

        # Chấm điểm Logic
        score, max_score, errs = grade_sheet_dynamic(student_answers, exam_config[page_num]["answer_key"])

        # Tổng hợp
        if student_id not in student_results:
            student_results[student_id] = {'Total_Score': 0, 'Total_Max': 0, 'Pages_Done': []}

        student_results[student_id]['Total_Score'] += score
        student_results[student_id]['Total_Max'] += max_score
        student_results[student_id]['Pages_Done'].append(page_num)

    # BƯỚC 3: XUẤT BÁO CÁO
    final_report = []
    required_pages = set(exam_config.keys())

    for sid, data in student_results.items():
        processed_pages = set(data['Pages_Done'])
        status = "Hợp lệ" if not required_pages - processed_pages else f"Thiếu trang"

        final_report.append({
            'SBD': sid,
            'Điểm Tổng': data['Total_Score'],
            'Điểm Tối Đa': data['Total_Max'],
            'Trạng Thái': status
        })

    df = pd.DataFrame(final_report).sort_values(by='SBD')
    df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(df)

In [39]:
if __name__ == '__main__':
    main()

--- HỆ THỐNG CHẤM THI OMR (EFFICIENTNET-B0) ---
 >> Đang quét Bản đồ Layout Trang 1...
--- Phân tích ảnh mẫu: modelAnswer_exam5_1.png ---
   > Mục tiêu: Cần tìm 60 ô.
   > Conf=0.5: Raw=60 -> FilterArea=60 -> FilterContainer=60
   ✅ Đã tìm thấy ĐỦ số lượng ô!
   SBD  Điểm Tổng  Điểm Tối Đa Trạng Thái
3  001        7.0         20.0     Hợp lệ
6  002        6.0         20.0     Hợp lệ
0  003        9.0         20.0     Hợp lệ
5  004        7.0         20.0     Hợp lệ
7  005        7.0         20.0     Hợp lệ
9  006        6.0         20.0     Hợp lệ
4  007        7.0         20.0     Hợp lệ
8  008        9.0         20.0     Hợp lệ
2  009        7.0         20.0     Hợp lệ
1  010       11.0         20.0     Hợp lệ
